In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['MODEL_API_VERSION'] = os.getenv('MODEL_API_VERSION')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

Em4f4T5UbT
gpt-5-nano


In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment = os.environ["MODEL_NAME"],
    azure_endpoint = os.environ["END_POINT"],
    openai_api_version = os.environ["MODEL_API_VERSION"],
    openai_api_key = os.environ["AZURE_OPENAI_API_KEY"],
)

In [3]:
import asyncio
import importlib
import time
from typing import Any, List

import utils
importlib.reload(utils)  # 수정된 save_transcript 반영 (PermissionError 폴백)
from utils import (
    CUSTOMER_BASE,
    SUPPORT_BASE,
    build_client,
    send_to_support,
    ask_customer_followup,
    save_transcript,
)


In [4]:
print(f"Customer: {CUSTOMER_BASE}, Support: {SUPPORT_BASE}")

Customer: http://127.0.0.1:8010, Support: http://127.0.0.1:8011


In [5]:
# 8010, 8011 포트 사용 중인 프로세스 정리 (macOS/Linux 공통)
import sys, subprocess, platform, time

def kill_port(port: int) -> None:
    if platform.system() == "Darwin":  # macOS
        subprocess.run(
            f"lsof -ti:{port} | xargs kill -9 2>/dev/null",
            shell=True, capture_output=True,
        )
    else:
        subprocess.run(
            f"fuser -k {port}/tcp 2>/dev/null",
            shell=True, capture_output=True,
        )

for port in [8010, 8011]:
    kill_port(port)
time.sleep(1)  # 종료 반영 대기

print("using python:", sys.executable)
subprocess.Popen([sys.executable, "a2a_support_server.py"])
subprocess.Popen([sys.executable, "a2a_customer_server.py"])
# 서버 기동 대기 (바로 build_client 하면 503 연결 실패 방지)
time.sleep(3)
print("서버 기동 대기 완료. 다음 셀에서 build_client 실행하세요.")

using python: /Users/ktjmh/MyData/GitHub/ktaiagent/.ktaiagent/bin/python
상담원 에이전트가 빌드 되었습니다.
고객 에이전트가 빌드 되었습니다.


INFO:     Started server process [53896]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8010 (Press CTRL+C to quit)


서버 기동 대기 완료. 다음 셀에서 build_client 실행하세요.


In [7]:
customer, customer_http = await build_client(CUSTOMER_BASE)
support,  support_http  = await build_client(SUPPORT_BASE)

INFO:     127.0.0.1:54013 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:54014 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


In [8]:
start_question = "요금제 변경할 때 위약금이 발생하는 기준을 알려주세요."
sup = await send_to_support(support, start_question)
print("상담원 ▶", sup)

INFO:     127.0.0.1:54014 - "POST / HTTP/1.1" 200 OK
상담원 ▶ 요금제 변경 시 발생하는 위약금은 약정 할인반환금으로 부과되며, 할인반환금 할인율을 적용해 산정됩니다.

기준(약정기간별 할인반환금 할인율 표에 따른 경과 기간에 따라)  
- 약정기간: 1년
  - 계약 후 0~6개월 이내: 0%
  - 계약 후 7~9개월 이내: 20%
  - 계약 후 10~12개월 이내: 130%
- 약정기간: 2년
  - 계약 후 0~6개월 이내: 0%
  - 계약 후 7~12개월 이내: 60%
  - 계약 후 13~16개월 이내: 95%
  - 계약 후 17~20개월 이내: 140%
  - 계약 후 21~24개월 이내: 180%
- 약정기간: 3년
  - 계약 후 0~6개월 이내: 0%
  - 계약 후 7~12개월 이내: 30%
  - 계약 후 13~16개월 이내: 65%
  - 계약 후 17~20개월 이내: 75%
  - 계약 후 21~24개월 이내: 100%
  - 계약 후 25~28개월 이내: 110%
  - 계약 후 29~32개월 이내: 125%

참고로: 요금제 월정액 할인은 3년 약정 시에만 가능하며, 회선당 2,750원까지 할인 가능하다는 내용도 있습니다.

질문 드립니다:
- 현재 약정기간은 몇 년이며, 남은 약정기간은 어느 정도인가요?
- 현재까지 받으신 약정 할인액의 총액(할인액 합계)을 알고 계신가요?
이 두 정보를 알려주시면 위약금(할인반환금)의 구체적인 금액을 함께 계산해 드리겠습니다.

참고 출처
- InternetServiceToU.pdf, 페이지 87, 153, 164, 159, 28 (약정기간별 할인반환금 할인율 및 관련 내용)


In [9]:
cust = await ask_customer_followup(customer, sup)
print("고객 ▶", cust)

INFO:     127.0.0.1:54023 - "POST / HTTP/1.1" 200 OK
고객 ▶ 현재 약정기간이 몇 년인지와 남은 약정기간이 몇 개월인지 알려주실 수 있을까요? 그리고 지금까지 받으신 약정 할인액의 총합도 함께 알려주시면 정확한 위약금(할인반환금)을 계산해 드리겠습니다.


In [10]:
sup = await send_to_support(support, start_question)
print("상담원 ▶", sup)

INFO:     127.0.0.1:54026 - "POST / HTTP/1.1" 200 OK
상담원 ▶ 다음 기준으로 위약금이 산정됩니다. (출처: InternetServiceToU.pdf, 페이지 87, 153, 159, 165)

- 위약금 산정 방식
  - 약정기간별 할인반환금 할인율을 적용해 위약금을 부과합니다.
  - 할인율은 약정 이용기간(약정 기간 동안 실제 사용한 기간)에 따라 다르게 적용됩니다.

- 요금제별 위약금 할인율 표 (약정기간/약정 이용기간/할인율)
  - 1년 약정
    - 계약 후 0~6개월 이내: 0%
    - 계약 후 7~9개월 이내: 20%
    - 계약 후 10~12개월 이내: 130%
  - 2년 약정
    - 계약 후 0~6개월 이내: 0%
    - 계약 후 7~12개월 이내: 60%
    - 계약 후 13~16개월 이내: 95%
    - 계약 후 17~20개월 이내: 140%
    - 계약 후 21~24개월 이내: 180%
  - 3년 약정
    - 계약 후 0~6개월 이내: 0%
    - 계약 후 7~12개월 이내: 30%
    - 계약 후 13~16개월 이내: 65%
    - 계약 후 17~20개월 이내: 75%
    - 계약 후 21~24개월 이내: 100%
    - 계약 후 25~28개월 이내: 110%
    - 계약 후 29~32개월 이내: 125%

- 참고
  - 약정기간별 총 할인금액과 관련한 내용도 함께 안내되며, 약정 이용기간별로 제공받은 약정 할인액의 합산금액에 대해 위약금을 부과합니다.
  - 일부 케이스에서 월정액 할인 등은 조건에 따라 다르게 적용될 수 있습니다.

추가로 필요한 정보가 있나요?
- 예를 들어 현재 약정 기간 남은 기간이나 변경하려는 요금제의 상세 조건을 알려주시면, 구체적으로 예상 위약금 금액을 계산해 드릴 수 있습니다.


In [11]:
topics = [
    "와이파이 요금제에 가입하고 싶어요",
    "요금제 변경 시 위약금이 어떻게 되나요?",
    "해외 로밍 데이터 한도가 있나요?",
    "유심을 교체하고 번호를 바꾸고 싶어요",
]
max_rounds = 3
transcript: List[dict] = []

try:
    for topic in topics:
        print(f"\n[TOPIC] {topic}")
        transcript.append({"who": "customer", "text": topic})

        support_reply = await send_to_support(support, topic)
        transcript.append({"who": "support", "text": support_reply})
        print(f" 상담원 ▶ {support_reply[:120]}...")

        for _ in range(1, max_rounds + 1):
            customer_follow = await ask_customer_followup(customer, support_reply)
            transcript.append({"who": "customer", "text": customer_follow})
            print(f" 고객 ▶ {customer_follow[:120]}...")

            if "감사합니다" in customer_follow:
                break

            support_reply = await send_to_support(support, customer_follow)
            transcript.append({"who": "support", "text": support_reply})
            print(f" 상담원 ▶ {support_reply[:120]}...")

        print("[DONE] topic finished.")
        print("=" * 70)
except Exception as e:
    print(f"오류 발생: {e}")

save_transcript(transcript, prefix="a2a-sim")


[TOPIC] 와이파이 요금제에 가입하고 싶어요
INFO:     127.0.0.1:54026 - "POST / HTTP/1.1" 200 OK
 상담원 ▶ 도와드리려면 현재 제공된 컨텍스트(표나 본문)가 없어 구체적인 정보를 확인할 수 없습니다.

다음 정보를 알려주시면 맞춤 안내를 드리겠습니다.
- 어느 유형의 와이파이 요금제인지: 가정용 인터넷(집가정용)인지, 이동...
INFO:     127.0.0.1:54041 - "POST / HTTP/1.1" 200 OK
 고객 ▶ 현재 이용 중인 와이파이 요금제의 유형이 가정용 인터넷인지, 이동통신사 Wi‑Fi 서비스인지 어느 쪽인지 먼저 알려주실 수 있을까요?...
INFO:     127.0.0.1:54044 - "POST / HTTP/1.1" 200 OK
 상담원 ▶ 죄송하지만 현재 대화의 컨텍스트만으로는 고객님의 와이파이 요금제 유형이 가정용 인터넷인지 이동통신사 Wi‑Fi 서비스인지 확정해 드릴 수 없습니다.

확인 방법 제안
- 계정/청구서의 요금제명이나 서비스 유형 표기를...
INFO:     127.0.0.1:54050 - "POST / HTTP/1.1" 200 OK
 고객 ▶ 계정에 연결된 전화번호를 알려드려도 바로 구분해 주실 수 있나요?...
INFO:     127.0.0.1:54053 - "POST / HTTP/1.1" 200 OK
 상담원 ▶ 제공된 컨텍스트에서 해당 기능의 가능 여부나 절차를 확인할 수 있는 정보가 없습니다. 

확인하고 싶은 계정의 서비스명과 현재 어떤 정보를 원하시는지 구체적으로 알려주시면, 그 범위 내에서 가능한 방법을 안내해 드리...
INFO:     127.0.0.1:54061 - "POST / HTTP/1.1" 200 OK
 고객 ▶ 계정의 서비스명이 무엇이며, 구체적으로 어떤 정보를 원하시는지요? 예를 들어 요금 내역이나 이용 현황, 기능 가능 여부 중 어떤 부분을 확인하고 싶으신지와, 본인 인증이 필요한지 여부도 함께 알려주실 수 있을까요?...
IN